In [1]:
import yfinance as yf
import pandas as pd
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS",
           "HDFCBANK.NS", "SBIN.NS"]

all_data = {}

for ticker in tickers:
    df = yf.download(ticker, start="2018-01-01",
                     end="2026-12-06", auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["tomorrow_close"] = df["Close"].shift(-1)
    df["target"] = (df["tomorrow_close"] > df["Close"]).astype(int)

    df.dropna(inplace=True)

    df["ticker"] = ticker

    all_data[ticker] = df
    print(f"{ticker}: {len(df)} rows | UP: {df['target'].mean()*100:.1f}%")

[*********************100%***********************]  1 of 1 completed


RELIANCE.NS: 2086 rows | UP: 51.5%


[*********************100%***********************]  1 of 1 completed


TCS.NS: 2086 rows | UP: 50.4%


[*********************100%***********************]  1 of 1 completed


INFY.NS: 2086 rows | UP: 52.0%


[*********************100%***********************]  1 of 1 completed


HDFCBANK.NS: 2086 rows | UP: 50.9%


[*********************100%***********************]  1 of 1 completed

SBIN.NS: 2086 rows | UP: 52.5%


In [2]:
final_df = pd.concat(all_data.values())
final_df.reset_index(inplace=True)
final_df.rename(columns={"index": "Date"}, inplace=True)

print(f"\nTotal rows: {len(final_df)}")
print(f"Columns: {list(final_df.columns)}")

final_df.to_csv("all_stocks_raw.csv", index=False)
print("\nSaved all_stocks_raw.csv")


Total rows: 10430
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'tomorrow_close', 'target', 'ticker']

Saved all_stocks_raw.csv


In [3]:
import ta
import numpy as np
import talib

def add_candlestick_patterns(df):
    o = df["Open"].values
    h = df["High"].values
    l = df["Low"].values
    c = df["Close"].values

    df["cdl_doji"]            = talib.CDLDOJI(o, h, l, c)
    df["cdl_hammer"]          = talib.CDLHAMMER(o, h, l, c)
    df["cdl_engulfing"]       = talib.CDLENGULFING(o, h, l, c)
    df["cdl_shooting_star"]   = talib.CDLSHOOTINGSTAR(o, h, l, c)
    df["cdl_morning_star"]    = talib.CDLMORNINGSTAR(o, h, l, c)
    df["cdl_evening_star"]    = talib.CDLEVENINGSTAR(o, h, l, c)
    df["cdl_marubozu"]        = talib.CDLMARUBOZU(o, h, l, c)
    df["cdl_3white_soldiers"] = talib.CDL3WHITESOLDIERS(o, h, l, c)

    # normalize 100/-100/0 → 1/-1/0
    pattern_cols = [col for col in df.columns if col.startswith("cdl_")]
    df[pattern_cols] = df[pattern_cols] / 100

    # combined signal features
    df["pattern_strength"] = df[pattern_cols].sum(axis=1)
    df["pattern_count"]    = (df[pattern_cols] != 0).sum(axis=1)

    return df

def add_features(df):

    # 1. MOMENTUM
    df["rsi"] = ta.momentum.RSIIndicator(
        df["Close"], window=14).rsi()

    #  2. MACD 
    macd = ta.trend.MACD(
        df["Close"], window_fast=12, window_slow=26, window_sign=9)
    df["macd"]        = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    df["macd_hist"]   = macd.macd_diff()

    #  3. EMA 
    df["ema_20"]   = ta.trend.EMAIndicator(
        df["Close"], window=20).ema_indicator()
    df["ema_50"]   = ta.trend.EMAIndicator(
        df["Close"], window=50).ema_indicator()
    df["ema_cross"] = (df["ema_20"] > df["ema_50"]).astype(int)

    #  4. BOLLINGER BANDS 
    bb = ta.volatility.BollingerBands(
        df["Close"], window=20, window_dev=2)
    df["bb_upper"] = bb.bollinger_hband()
    df["bb_lower"] = bb.bollinger_lband()
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

    #  5. ATR 
    df["atr"] = ta.volatility.AverageTrueRange(
        df["High"], df["Low"], df["Close"], window=14
    ).average_true_range()

    #  6. VOLUME 
    df["volume_ma"]    = df["Volume"].rolling(20).mean()
    df["volume_ratio"] = df["Volume"] / df["volume_ma"]

    # 7. RETURN / LAG FEATURES 
    df["return_1d"]  = df["Close"].pct_change(1)
    df["return_5d"]  = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)
    df["lag_1"]      = df["return_1d"].shift(1)
    df["lag_2"]      = df["return_1d"].shift(2)
    df["lag_3"]      = df["return_1d"].shift(3)

    #  8. CANDLESTICK PATTERNS 
    df = add_candlestick_patterns(df)
 
    df.dropna(inplace=True)

    return df


for ticker in tickers:
    all_data[ticker] = add_features(all_data[ticker])
    print(f"{ticker}: {len(all_data[ticker])} rows, "
          f"{len(all_data[ticker].columns)} columns")
    

final_df = pd.concat(all_data.values())
final_df.reset_index(inplace=True)
final_df.rename(columns={"index": "Date"}, inplace=True)

print(f"Total rows     : {len(final_df)}")
print(f"Total columns  : {len(final_df.columns)}")
print(f"Columns        : {list(final_df.columns)}")

final_df.to_csv("all_stocks_features.csv", index=False)
print("\nSaved all_stocks_features.csv")


nan_check = final_df.isnull().sum()
print(nan_check[nan_check > 0])


print(f"\nClean rows : {len(final_df)}")
print(f"UP days    : {final_df['target'].mean()*100:.1f}%")

RELIANCE.NS: 2037 rows, 37 columns
TCS.NS: 2037 rows, 37 columns
INFY.NS: 2037 rows, 37 columns
HDFCBANK.NS: 2037 rows, 37 columns
SBIN.NS: 2037 rows, 37 columns
Total rows     : 10185
Total columns  : 38
Columns        : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'tomorrow_close', 'target', 'ticker', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'ema_20', 'ema_50', 'ema_cross', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_ma', 'volume_ratio', 'return_1d', 'return_5d', 'return_10d', 'lag_1', 'lag_2', 'lag_3', 'cdl_doji', 'cdl_hammer', 'cdl_engulfing', 'cdl_shooting_star', 'cdl_morning_star', 'cdl_evening_star', 'cdl_marubozu', 'cdl_3white_soldiers', 'pattern_strength', 'pattern_count']

Saved all_stocks_features.csv
Series([], dtype: int64)

Clean rows : 10185
UP days    : 51.5%
